In [14]:
import pandas as pd
import math

In [15]:
# change the file path according to your file's location
file = "/content/inpatient_admissions.csv"
df = pd.read_csv(file)
df.head()

,admission_id,patient_id,admission_date,discharge_date
0,ADM00001,P0092,2015-01-07,2015-01-08
1,ADM00002,P0070,2015-01-18,2015-01-19
2,ADM00003,P0127,2015-01-19,2015-01-20
3,ADM00004,P0003,2015-01-23,2015-01-24
4,ADM00005,P0085,2015-01-23,2015-01-24


In [18]:
# getting a summary of the dataset
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 623 entries, 0 to 622
Data columns (total 4 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   admission_id    623 non-null    object
 1   patient_id      623 non-null    object
 2   admission_date  623 non-null    object
 3   discharge_date  623 non-null    object
dtypes: object(4)
memory usage: 19.6+ KB


In [5]:
# converting admission_date and discharge_date to datetime
df['admission_date'] = pd.to_datetime(df['admission_date'])
df['discharge_date'] = pd.to_datetime(df['discharge_date'])
df.dtypes

,0
admission_id,object
patient_id,object
admission_date,datetime64[ns]
discharge_date,datetime64[ns]


In [6]:
# assign rank to rows sorted by admission date
df['rank'] = df.groupby('patient_id')['admission_date'].rank(ascending=True, method='min')
df.sort_values(by=['patient_id', 'rank'])

,admission_id,patient_id,admission_date,discharge_date,rank
83,ADM00084,P0001,2016-01-19,2016-01-20,1.0
255,ADM00256,P0001,2018-06-19,2018-06-20,2.0
108,ADM00109,P0002,2016-05-27,2016-05-28,1.0
188,ADM00189,P0002,2017-08-04,2017-08-05,2.0
222,ADM00223,P0002,2018-01-19,2018-01-20,3.0
...,...,...,...,...,...
565,ADM00566,P0127,2023-04-26,2023-04-27,9.0
580,ADM00581,P0127,2024-04-26,2024-04-27,10.0
614,ADM00615,P0127,2025-05-10,2025-05-11,11.0
620,ADM00621,P0127,2025-11-06,2025-11-07,12.0


In [7]:
# getting previous admission's discharge data in subsequent admission
df['prev_discharge_date'] = df.groupby('patient_id')['discharge_date'].shift(1)
df.sort_values(by=['patient_id', 'rank'])

,admission_id,patient_id,admission_date,discharge_date,rank,prev_discharge_date
83,ADM00084,P0001,2016-01-19,2016-01-20,1.0,NaT
255,ADM00256,P0001,2018-06-19,2018-06-20,2.0,2016-01-20
108,ADM00109,P0002,2016-05-27,2016-05-28,1.0,NaT
188,ADM00189,P0002,2017-08-04,2017-08-05,2.0,2016-05-28
222,ADM00223,P0002,2018-01-19,2018-01-20,3.0,2017-08-05
...,...,...,...,...,...,...
565,ADM00566,P0127,2023-04-26,2023-04-27,9.0,2022-05-26
580,ADM00581,P0127,2024-04-26,2024-04-27,10.0,2023-04-27
614,ADM00615,P0127,2025-05-10,2025-05-11,11.0,2024-04-27
620,ADM00621,P0127,2025-11-06,2025-11-07,12.0,2025-05-11


In [8]:
# get patients with multiple admissions i.e., rank > 1
multiple_admissions_df = df[df['rank'] > 1]
multiple_admissions_df.sort_values(by=['patient_id', 'rank'])

,admission_id,patient_id,admission_date,discharge_date,rank,prev_discharge_date
255,ADM00256,P0001,2018-06-19,2018-06-20,2.0,2016-01-20
188,ADM00189,P0002,2017-08-04,2017-08-05,2.0,2016-05-28
222,ADM00223,P0002,2018-01-19,2018-01-20,3.0,2017-08-05
585,ADM00586,P0002,2024-07-01,2024-07-02,4.0,2018-01-20
168,ADM00169,P0006,2017-04-09,2017-04-10,2.0,2016-10-22
...,...,...,...,...,...,...
530,ADM00531,P0127,2022-05-25,2022-05-26,8.0,2021-05-11
565,ADM00566,P0127,2023-04-26,2023-04-27,9.0,2022-05-26
580,ADM00581,P0127,2024-04-26,2024-04-27,10.0,2023-04-27
614,ADM00615,P0127,2025-05-10,2025-05-11,11.0,2024-04-27


In [9]:
# calculate the difference between previous discharge and admission_date
multiple_admissions_df['readmission_days'] = (multiple_admissions_df['admission_date'] - multiple_admissions_df['prev_discharge_date']).dt.days
multiple_admissions_df.sort_values(by=['patient_id', 'rank'])

/tmp/ipykernel_2103/2182501394.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  multiple_admissions_df['readmission_days'] = (multiple_admissions_df['admission_date'] - multiple_admissions_df['prev_discharge_date']).dt.days


,admission_id,patient_id,admission_date,discharge_date,rank,prev_discharge_date,readmission_days
255,ADM00256,P0001,2018-06-19,2018-06-20,2.0,2016-01-20,881
188,ADM00189,P0002,2017-08-04,2017-08-05,2.0,2016-05-28,433
222,ADM00223,P0002,2018-01-19,2018-01-20,3.0,2017-08-05,167
585,ADM00586,P0002,2024-07-01,2024-07-02,4.0,2018-01-20,2354
168,ADM00169,P0006,2017-04-09,2017-04-10,2.0,2016-10-22,169
...,...,...,...,...,...,...,...
530,ADM00531,P0127,2022-05-25,2022-05-26,8.0,2021-05-11,379
565,ADM00566,P0127,2023-04-26,2023-04-27,9.0,2022-05-26,335
580,ADM00581,P0127,2024-04-26,2024-04-27,10.0,2023-04-27,365
614,ADM00615,P0127,2025-05-10,2025-05-11,11.0,2024-04-27,378


In [10]:
# checking if the admission was a re-admission
multiple_admissions_df['is_readmission'] = multiple_admissions_df['readmission_days'] <= 30
multiple_admissions_df.sort_values(by=['patient_id', 'rank'])

/tmp/ipykernel_2103/912867271.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  multiple_admissions_df['is_readmission'] = multiple_admissions_df['readmission_days'] <= 30


,admission_id,patient_id,admission_date,discharge_date,rank,prev_discharge_date,readmission_days,is_readmission
255,ADM00256,P0001,2018-06-19,2018-06-20,2.0,2016-01-20,881,False
188,ADM00189,P0002,2017-08-04,2017-08-05,2.0,2016-05-28,433,False
222,ADM00223,P0002,2018-01-19,2018-01-20,3.0,2017-08-05,167,False
585,ADM00586,P0002,2024-07-01,2024-07-02,4.0,2018-01-20,2354,False
168,ADM00169,P0006,2017-04-09,2017-04-10,2.0,2016-10-22,169,False
...,...,...,...,...,...,...,...,...
530,ADM00531,P0127,2022-05-25,2022-05-26,8.0,2021-05-11,379,False
565,ADM00566,P0127,2023-04-26,2023-04-27,9.0,2022-05-26,335,False
580,ADM00581,P0127,2024-04-26,2024-04-27,10.0,2023-04-27,365,False
614,ADM00615,P0127,2025-05-10,2025-05-11,11.0,2024-04-27,378,False


In [11]:
re_admissions_df = multiple_admissions_df[multiple_admissions_df['is_readmission']]
re_admissions_df.sort_values(by=['patient_id', 'rank'])

,admission_id,patient_id,admission_date,discharge_date,rank,prev_discharge_date,readmission_days,is_readmission
285,ADM00286,P0016,2018-10-15,2018-10-16,2.0,2018-09-16,29,True
302,ADM00303,P0016,2018-12-17,2018-12-18,4.0,2018-11-17,30,True
322,ADM00323,P0016,2019-02-19,2019-02-20,6.0,2019-01-21,29,True
330,ADM00331,P0016,2019-03-20,2019-03-21,7.0,2019-02-20,28,True
338,ADM00339,P0016,2019-04-19,2019-04-20,8.0,2019-03-21,29,True
...,...,...,...,...,...,...,...,...
480,ADM00481,P0116,2021-05-29,2021-05-30,40.0,2021-05-02,27,True
499,ADM00500,P0116,2021-09-03,2021-09-04,43.0,2021-08-04,30,True
503,ADM00504,P0116,2021-10-04,2021-10-05,44.0,2021-09-04,30,True
506,ADM00507,P0116,2021-11-02,2021-11-03,45.0,2021-10-05,28,True


In [12]:
# calculate the re-admissions rate = re-admissions/total admissions
re_admissions = re_admissions_df.nunique()['admission_id']
total_admissions = df.nunique()['admission_id']
# round down the admissions rate
re_admissions_rate = math.floor((re_admissions / total_admissions) * 100)
print(f"Re-admissions Rate: {re_admissions_rate}")

Re-admissions Rate: 37
